# NOAA AIS 2015–2024 full download and port clipping (Google Colab)

This standalone Colab notebook downloads every available NOAA daily national AIS CSV archive for 2015–2024, immediately clips cargo/tanker observations to the configured major-port study areas, writes one Parquet file per day, and packages one ZIP per year in Google Drive.

It deliberately does **not** retain the national raw archives: NOAA reports that a single year can exceed 100 GB. Only one raw day is held in Colab temporary storage at a time and is deleted after its clipped Parquet has been copied to Drive.

The workflow is restartable. Completed daily Parquet files and completed annual ZIPs are skipped. If Colab disconnects, reconnect and run all cells again.

Annual ZIP contents use the exact structure expected by the local analysis notebook:

```text
year=2015/AIS_2015_01_01.parquet
year=2015/AIS_2015_01_02.parquet
...
```

The `arthur_kill_south` gate reuses the broad `new_york_harbor` study area, so it does not require a duplicated point layer. Section 10 in the updated local notebook handles that mapping.


In [ ]:
%pip install -q pyarrow beautifulsoup4 requests tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Annual ZIPs, resumable daily staging files, manifests, and status logs live here.
DRIVE_ROOT = Path('/content/drive/MyDrive/supply-chain-resilience/ais_major_ports_2015_2024')
STAGING_ROOT = DRIVE_ROOT / 'daily_staging'
STATUS_ROOT = DRIVE_ROOT / 'status'
LOCAL_WORK = Path('/content/ais_work')
for folder in (DRIVE_ROOT, STAGING_ROOT, STATUS_ROOT, LOCAL_WORK):
    folder.mkdir(parents=True, exist_ok=True)

START_YEAR = 2015
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))

# Set to a small integer (for example 3) for a pilot. None processes all missing days.
MAX_NEW_DAYS_PER_SESSION = None

# Existing annual ZIP + SHA256 files are skipped unless this is True.
REBUILD_COMPLETED_YEARS = False

# Keep staging by default because it makes recovery and auditing easy.
# Change to True only after verifying each annual ZIP.
DELETE_DRIVE_STAGING_AFTER_ZIP = False

CSV_CHUNK_ROWS = 250_000
REQUEST_TIMEOUT_SECONDS = 300
KEEP_CARGO_AND_TANKER_ONLY = True
STRICT_COMPLETE_CALENDAR = True
NOAA_ROOT = 'https://coast.noaa.gov/htdata/CMSP/AISDataHandler/'
USER_AGENT = 'supply-chain-resilience-research/0.1 (Colab NOAA AIS downloader)'

print(f'Drive output: {DRIVE_ROOT}')
print(f'Years: {START_YEAR}–{END_YEAR}')


## Port study areas

These are the same processing bboxes used by `ais_daily_download_2010_2025.ipynb`. Nearby physical gates may deliberately share or overlap a bbox. `arthur_kill_south` is not listed separately because its AIS source is `new_york_harbor`.


In [ ]:
STUDY_BBOXES = {
    'los_angeles_entrance': (-118.38, 33.62, -118.08, 33.88),
    'long_beach_entrance': (-118.28, 33.62, -117.98, 33.88),
    'oakland_entrance': (-122.48, 37.68, -122.18, 37.92),
    'portland_or_river': (-123.05, 45.35, -122.45, 45.82),
    'seattle_harbor': (-122.52, 47.42, -122.22, 47.78),
    'tacoma_harbor': (-122.68, 47.10, -122.25, 47.43),
    'new_york_harbor': (-74.28, 40.42, -73.72, 40.88),
    'delaware_river_south': (-75.75, 39.35, -74.92, 40.18),
    'baltimore_harbor': (-76.78, 39.00, -76.28, 39.42),
    'hampton_roads_entrance': (-76.62, 36.70, -75.72, 37.20),
    'charleston_entrance': (-80.12, 32.58, -79.58, 33.00),
    'savannah_river_entrance': (-81.28, 31.82, -80.62, 32.32),
    'tampa_bay_entrance': (-82.90, 27.35, -82.25, 28.15),
    'jacksonville_entrance': (-81.78, 30.12, -81.12, 30.62),
    'miami_entrance': (-80.38, 25.58, -79.92, 25.96),
    'port_everglades_entrance': (-80.32, 25.92, -79.90, 26.26),
    'mobile_bay_channel': (-88.30, 30.35, -87.75, 30.92),
    'new_orleans_river': (-90.40, 29.60, -89.65, 30.22),
    'galveston_bay_entrance': (-95.20, 29.00, -94.35, 29.80),
}

print(f'{len(STUDY_BBOXES)} stored AIS study areas for 20 physical gates.')


In [ ]:
from __future__ import annotations

import hashlib
import json
import re
import shutil
import time
import zipfile
from datetime import date, datetime
from urllib.parse import urljoin, urlparse

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

OUTPUT_COLUMNS = [
    'MMSI', 'BaseDateTime', 'event_date', 'LAT', 'LON', 'SOG', 'COG',
    'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status',
    'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass',
    'vessel_group', 'study_area',
]
NUMERIC_FLOAT_COLUMNS = ['LAT', 'LON', 'SOG', 'COG', 'Heading', 'Length', 'Width', 'Draft']
STRING_COLUMNS = ['MMSI', 'VesselName', 'IMO', 'CallSign', 'Status', 'TransceiverClass']
ALIASES = {
    'mmsi': 'MMSI', 'basedatetime': 'BaseDateTime', 'base_date_time': 'BaseDateTime',
    'latitude': 'LAT', 'lat': 'LAT', 'longitude': 'LON', 'lon': 'LON',
    'sog': 'SOG', 'cog': 'COG', 'heading': 'Heading',
    'vesselname': 'VesselName', 'vessel_name': 'VesselName',
    'imo': 'IMO', 'callsign': 'CallSign', 'call_sign': 'CallSign',
    'vesseltype': 'VesselType', 'vessel_type': 'VesselType',
    'status': 'Status', 'length': 'Length', 'width': 'Width',
    'draft': 'Draft', 'cargo': 'Cargo', 'transceiverclass': 'TransceiverClass',
}

OUTPUT_SCHEMA = pa.schema([
    pa.field('MMSI', pa.string()),
    pa.field('BaseDateTime', pa.timestamp('ns', tz='UTC')),
    pa.field('event_date', pa.date32()),
    pa.field('LAT', pa.float64()), pa.field('LON', pa.float64()),
    pa.field('SOG', pa.float64()), pa.field('COG', pa.float64()),
    pa.field('Heading', pa.float64()), pa.field('VesselName', pa.string()),
    pa.field('IMO', pa.string()), pa.field('CallSign', pa.string()),
    pa.field('VesselType', pa.int64()), pa.field('Status', pa.string()),
    pa.field('Length', pa.float64()), pa.field('Width', pa.float64()),
    pa.field('Draft', pa.float64()), pa.field('Cargo', pa.int64()),
    pa.field('TransceiverClass', pa.string()),
    pa.field('vessel_group', pa.string()), pa.field('study_area', pa.string()),
])

def _ts() -> str:
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

def canonicalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for column in frame.columns:
        key = re.sub(r'[^a-z0-9_]', '', str(column).strip().lower())
        if key in ALIASES:
            rename[column] = ALIASES[key]
    return frame.rename(columns=rename)

def clip_and_normalize(frame: pd.DataFrame) -> pd.DataFrame:
    frame = canonicalize_columns(frame.copy())
    if 'LAT' not in frame or 'LON' not in frame:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    frame['LAT'] = pd.to_numeric(frame['LAT'], errors='coerce')
    frame['LON'] = pd.to_numeric(frame['LON'], errors='coerce')
    frame = frame[frame['LAT'].between(-90, 90) & frame['LON'].between(-180, 180)]

    clipped = []
    for area, (min_lon, min_lat, max_lon, max_lat) in STUDY_BBOXES.items():
        keep = frame['LON'].between(min_lon, max_lon) & frame['LAT'].between(min_lat, max_lat)
        if keep.any():
            part = frame.loc[keep].copy()
            part['study_area'] = area
            clipped.append(part)
    if not clipped:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    frame = pd.concat(clipped, ignore_index=True)

    for column in ('VesselType', 'Cargo'):
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Int64')

    cargo_codes_avis = {1003, 1004, 1016}
    tanker_codes_avis = {1017, 1024}
    cargo_mask = (
        frame['VesselType'].between(70, 79)
        | frame['VesselType'].isin(cargo_codes_avis)
        | frame['Cargo'].between(70, 79)
    )
    tanker_mask = (
        frame['VesselType'].between(80, 89)
        | frame['VesselType'].isin(tanker_codes_avis)
        | frame['Cargo'].between(80, 89)
    )
    frame['vessel_group'] = pd.Series('unknown', index=frame.index, dtype='string')
    frame.loc[cargo_mask, 'vessel_group'] = 'cargo'
    frame.loc[tanker_mask, 'vessel_group'] = 'tanker'
    if KEEP_CARGO_AND_TANKER_ONLY:
        frame = frame[frame['vessel_group'].isin(['cargo', 'tanker'])]

    if 'BaseDateTime' not in frame:
        frame['BaseDateTime'] = pd.NaT
    frame['BaseDateTime'] = pd.to_datetime(frame['BaseDateTime'], errors='coerce', utc=True)
    frame['event_date'] = frame['BaseDateTime'].dt.date

    for column in NUMERIC_FLOAT_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Float64')
    for column in STRING_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
        frame[column] = frame[column].astype('string')
    for column in OUTPUT_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    return frame[OUTPUT_COLUMNS].reset_index(drop=True)

print(f'[{_ts()}] Processing functions ready.')


In [ ]:
retry = Retry(
    total=4, connect=4, read=4, backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset({'GET', 'HEAD'}),
)
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': USER_AGENT})
SESSION.mount('https://', HTTPAdapter(max_retries=retry))
DAILY_PATTERN = re.compile(r'AIS_(\d{4})_(\d{2})_(\d{2})\.zip$', re.I)

def discover_year(year: int) -> list[dict]:
    index_url = f'{NOAA_ROOT}{year}/index.html'
    response = SESSION.get(index_url, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    records, seen = [], set()
    for link in soup.find_all('a', href=True):
        url = urljoin(index_url, link['href'])
        name = Path(urlparse(url).path).name
        match = DAILY_PATTERN.fullmatch(name)
        if not match or url in seen:
            continue
        seen.add(url)
        y, month, day = map(int, match.groups())
        archive_date = date(y, month, day)
        if y == year:
            records.append({
                'year': year, 'date': archive_date,
                'archive_name': name, 'url': url,
            })
    return sorted(records, key=lambda row: (row['date'], row['archive_name']))

manifest_records = []
for year in tqdm(YEARS, desc='Discovering NOAA yearly indexes'):
    manifest_records.extend(discover_year(year))
manifest = pd.DataFrame(manifest_records).sort_values(['date', 'archive_name']).reset_index(drop=True)
manifest_path = DRIVE_ROOT / f'noaa_daily_manifest_{START_YEAR}_{END_YEAR}.csv'
manifest.to_csv(manifest_path, index=False)

expected_dates = set(pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31', freq='D').date)
manifest_dates = set(manifest['date'])
missing_dates = sorted(expected_dates - manifest_dates)
duplicate_dates = manifest['date'].value_counts().loc[lambda counts: counts > 1]
summary = manifest.groupby('year').size().rename('archives').to_frame()
summary['expected_calendar_days'] = [366 if pd.Timestamp(f'{year}-12-31').is_leap_year else 365 for year in summary.index]
display(summary)
print(f'Manifest rows: {len(manifest):,}')
print(f'Missing calendar dates: {len(missing_dates):,}; duplicated dates: {len(duplicate_dates):,}')
if missing_dates:
    display(pd.DataFrame({'missing_date': missing_dates[:50]}))
if STRICT_COMPLETE_CALENDAR and (missing_dates or len(duplicate_dates)):
    raise RuntimeError('NOAA manifest is not one archive per calendar day. Review the displayed gaps before downloading.')


In [ ]:
def download_archive(record: dict, attempts: int = 4) -> Path:
    destination = LOCAL_WORK / record['archive_name']
    partial = destination.with_suffix('.zip.part')
    if destination.exists() and zipfile.is_zipfile(destination):
        return destination

    for attempt in range(attempts):
        existing = partial.stat().st_size if partial.exists() else 0
        headers = {'Range': f'bytes={existing}-'} if existing else {}
        try:
            with SESSION.get(record['url'], headers=headers, stream=True, timeout=REQUEST_TIMEOUT_SECONDS) as response:
                response.raise_for_status()
                if existing and response.status_code != 206:
                    partial.unlink(missing_ok=True)
                    existing = 0
                mode = 'ab' if existing and response.status_code == 206 else 'wb'
                remaining = int(response.headers.get('Content-Length', 0))
                total = existing + remaining if remaining else None
                with partial.open(mode) as handle, tqdm(
                    total=total, initial=existing, unit='B', unit_scale=True,
                    desc=record['archive_name'], leave=False,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                            progress.update(len(chunk))
            partial.replace(destination)
            if not zipfile.is_zipfile(destination):
                raise RuntimeError('Downloaded file is not a valid ZIP.')
            return destination
        except Exception:
            destination.unlink(missing_ok=True)
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError('Unreachable')

def csv_frames_from_zip(archive_path: Path):
    with zipfile.ZipFile(archive_path) as bundle:
        members = [name for name in bundle.namelist() if name.lower().endswith('.csv')]
        if not members:
            raise RuntimeError(f'No CSV found in {archive_path.name}')
        for member in members:
            with bundle.open(member) as handle:
                yield from pd.read_csv(
                    handle, chunksize=CSV_CHUNK_ROWS, low_memory=False,
                    on_bad_lines='skip',
                )

def write_daily_parquet(archive_path: Path, output_path: Path) -> int:
    temporary = output_path.with_suffix('.parquet.part')
    temporary.unlink(missing_ok=True)
    writer = pq.ParquetWriter(temporary, OUTPUT_SCHEMA, compression='zstd')
    rows = 0
    try:
        for frame in csv_frames_from_zip(archive_path):
            normalized = clip_and_normalize(frame)
            if normalized.empty:
                continue
            table = pa.Table.from_pandas(
                normalized, schema=OUTPUT_SCHEMA, preserve_index=False, safe=False,
            )
            writer.write_table(table)
            rows += len(normalized)
    finally:
        writer.close()
    temporary.replace(output_path)
    return rows

def valid_parquet(path: Path) -> bool:
    try:
        return path.exists() and path.stat().st_size > 0 and pq.read_metadata(path).num_columns == len(OUTPUT_COLUMNS)
    except Exception:
        return False

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def package_year(year: int, records: list[dict], year_dir: Path) -> Path:
    expected_names = {Path(record['archive_name']).stem + '.parquet' for record in records}
    actual_names = {path.name for path in year_dir.glob('AIS_*.parquet') if valid_parquet(path)}
    missing = sorted(expected_names - actual_names)
    if missing:
        raise RuntimeError(f'Cannot package {year}; {len(missing)} daily Parquet files are missing.')

    final_zip = DRIVE_ROOT / f'ais_points_major_ports_{year}.zip'
    final_sha = final_zip.with_suffix('.zip.sha256')
    local_zip = LOCAL_WORK / f'ais_points_major_ports_{year}.zip.part'
    local_zip.unlink(missing_ok=True)
    success = {
        'year': year, 'daily_files': len(expected_names),
        'created_at': datetime.now().isoformat(timespec='seconds'),
        'archive_layout': f'year={year}/AIS_{year}_MM_DD.parquet',
    }
    with zipfile.ZipFile(local_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
        for path in tqdm(sorted(year_dir.glob('AIS_*.parquet')), desc=f'Packaging {year}'):
            bundle.write(path, arcname=f'year={year}/{path.name}')
        bundle.writestr(f'year={year}/_SUCCESS.json', json.dumps(success, indent=2))
    if not zipfile.is_zipfile(local_zip):
        raise RuntimeError(f'Annual archive validation failed: {local_zip}')
    shutil.copy2(local_zip, final_zip)
    digest = sha256_file(final_zip)
    final_sha.write_text(f'{digest}  {final_zip.name}\n', encoding='utf-8')
    local_zip.unlink(missing_ok=True)
    return final_zip

print(f'[{_ts()}] Download, conversion, and packaging functions ready.')


## Run the full decade

This cell checkpoints after every successfully processed day. A failed day is logged and retried the next time the notebook runs. A year is packaged only when every manifest day has a valid Parquet file.


In [ ]:
new_days_this_session = 0
stop_requested = False
session_started = time.time()

for year in YEARS:
    records = manifest.loc[manifest['year'].eq(year)].to_dict('records')
    year_dir = STAGING_ROOT / f'year={year}'
    year_dir.mkdir(parents=True, exist_ok=True)
    status_path = STATUS_ROOT / f'ais_download_status_{year}.csv'
    final_zip = DRIVE_ROOT / f'ais_points_major_ports_{year}.zip'
    final_sha = final_zip.with_suffix('.zip.sha256')

    if final_zip.exists() and final_sha.exists() and not REBUILD_COMPLETED_YEARS:
        print(f'[{_ts()}] ✓ {year} annual ZIP already exists; skipping.')
        continue

    print(f'[{_ts()}] ── {year}: {len(records)} daily archives ──')
    statuses = []
    if status_path.exists():
        statuses = pd.read_csv(status_path).to_dict('records')

    for day_index, record in enumerate(records, 1):
        output_name = Path(record['archive_name']).stem + '.parquet'
        drive_output = year_dir / output_name
        if valid_parquet(drive_output):
            continue
        if MAX_NEW_DAYS_PER_SESSION is not None and new_days_this_session >= MAX_NEW_DAYS_PER_SESSION:
            stop_requested = True
            break

        archive_path = None
        local_output = LOCAL_WORK / output_name
        started = time.time()
        status = {
            'year': year, 'date': str(record['date']),
            'archive_name': record['archive_name'], 'status': 'pending',
            'rows': None, 'elapsed_minutes': None, 'error': None,
            'updated_at': datetime.now().isoformat(timespec='seconds'),
        }
        try:
            print(f'[{_ts()}] {year} day {day_index}/{len(records)}: {record["archive_name"]}')
            archive_path = download_archive(record)
            rows = write_daily_parquet(archive_path, local_output)
            shutil.copy2(local_output, drive_output)
            if not valid_parquet(drive_output):
                raise RuntimeError('Drive Parquet validation failed after copy.')
            status.update(status='processed', rows=rows)
            new_days_this_session += 1
            print(f'  ✓ {rows:,} clipped rows → {drive_output.name}')
        except Exception as exc:
            status.update(status='failed', error=f'{type(exc).__name__}: {exc}')
            print(f'  ✗ {status["error"]}')
        finally:
            status['elapsed_minutes'] = round((time.time() - started) / 60, 2)
            status['updated_at'] = datetime.now().isoformat(timespec='seconds')
            statuses = [row for row in statuses if row.get('archive_name') != record['archive_name']]
            statuses.append(status)
            pd.DataFrame(statuses).sort_values(['date', 'archive_name']).to_csv(status_path, index=False)
            if archive_path is not None:
                archive_path.unlink(missing_ok=True)
            local_output.unlink(missing_ok=True)

    if stop_requested:
        print(f'[{_ts()}] Pilot/session limit reached after {new_days_this_session} new days.')
        break

    expected = len(records)
    completed = sum(valid_parquet(year_dir / (Path(row['archive_name']).stem + '.parquet')) for row in records)
    if completed == expected:
        archive = package_year(year, records, year_dir)
        print(f'[{_ts()}] ✓ Packaged {year}: {archive} ({archive.stat().st_size / 1024**3:.2f} GiB)')
        if DELETE_DRIVE_STAGING_AFTER_ZIP:
            shutil.rmtree(year_dir)
            print(f'  Removed Drive staging directory: {year_dir}')
    else:
        print(f'[{_ts()}] {year} incomplete: {completed}/{expected} daily files; rerun to retry missing days.')

elapsed_hours = (time.time() - session_started) / 3600
print(f'[{_ts()}] Session finished: {new_days_this_session} new days in {elapsed_hours:.2f} hours.')


In [ ]:
verification_rows = []
for year in YEARS:
    archive = DRIVE_ROOT / f'ais_points_major_ports_{year}.zip'
    checksum = archive.with_suffix('.zip.sha256')
    year_dir = STAGING_ROOT / f'year={year}'
    expected = int(manifest['year'].eq(year).sum())
    staged = sum(valid_parquet(path) for path in year_dir.glob('AIS_*.parquet')) if year_dir.exists() else 0
    verification_rows.append({
        'year': year, 'expected_days': expected, 'staged_days': staged,
        'annual_zip': archive.exists(), 'sha256': checksum.exists(),
        'zip_GiB': round(archive.stat().st_size / 1024**3, 3) if archive.exists() else None,
    })
verification = pd.DataFrame(verification_rows)
display(verification)
if verification['annual_zip'].all() and verification['sha256'].all():
    print('All annual archives and checksum files are present.')
else:
    print('Some years are incomplete. Rerun the full-decade cell to resume.')


## Download and use locally

Download the ten files `ais_points_major_ports_2015.zip` through `ais_points_major_ports_2024.zip` from My Drive. Optionally verify each download against its `.sha256` file.

Extract every ZIP directly into the local directory below (not into ten extra nested folders):

```text
/Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/interim/ais_points_major_ports_2015_2024_v3/
```

After extraction the layout must be:

```text
ais_points_major_ports_2015_2024_v3/
├── year=2015/AIS_2015_01_01.parquet
├── year=2016/AIS_2016_01_01.parquet
...
└── year=2024/AIS_2024_12_31.parquet
```

Then open `ais_daily_download_2010_2025.ipynb`. Do not rerun its download/normalization pipeline. Rerun the configuration cell so it includes all 20 physical gates, then run Section 10, **Detect gateway crossings and build daily metrics**.

Section 10 will read `year=*/*.parquet`, use both `new_york_harbor` and `arthur_kill_south`, and write the crossing-event and daily port-group metric files under `data/processed/`.
